# 0.2 · The PD prior

*0. General · notebook 0.2 of the story.* ← [0.1 · the real credit data](<0.1_data_exploration.ipynb>) · [0.3 · the LGD prior](<0.3_prior_visualisation_lgd.ipynb>) →

**What does our credit prior generate for PD, how does it differ from TabICL's own, and does it look
like the real books of 0.1?** The target is an imbalanced binary default flag. Experiment 1 then trains on both priors and lets
the benchmark decide; this notebook checks, before that compute is spent, that the prior is worth
training on — that it looks like real data, that each credit mechanism produces the structure it
claims, and that the task it sets is the right difficulty.

Every figure generates live from `config/Exp1_PD.yaml` through `TaskGenerator` — the exact code path
training uses — or reads pre-generated pools when they are on disk. **Original** is the prior at
`credit_fraction = 0`: TabICL's `graph_scm`, unchanged, and exactly Experiment 1's control. **Credit** is
ours, at `credit_fraction = 1`.

**How to read it.** Part A compares the two priors against the real data. Part B takes our prior apart,
one credit mechanism at a time, to show where each difference comes from. Part C checks that the task is
learnable and the tables sane. Figures that answered nothing have been removed rather than kept for
volume.

## Grounding in the literature

PD is imbalanced binary classification. The prior this project runs is TabICLv2's `graph_scm`, which
makes the categorical target natively in-graph: imbalance enters only incidentally, through the softmax
converter's bias `b = log(w)` (`repositories/NanoTabICL.txt` `rand_converter`; `TabICL.txt`
`CategoricalConverter`; paper §E.6), and nothing there aims at a credit-like rare rate. Our credit path
adds that. O'Prior finds that **structural mechanism diversity, not observational realism**, drives
transfer (`papers/2026/05_Bouadi_ShapingThePrior` Table 2) — so Part A's realism is a sanity check, not
the claim. **Honesty flag:** the Merton/Vasicek one-factor default model and the Basel IRB asset
correlations behind the correlated-default mechanism are external credit-risk knowledge, **not in
`tfm-library`** (a whole-library search returns nothing); figures draw them in amber, and no TFM paper
is cited for them. Pin `e5ce016`.

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, pathlib
ROOT = pathlib.Path.cwd()
# Walk up to the repository root — the notebook may be opened from its chapter folder
# (notebooks/1. Experiment 1/), from notebooks/, or from the root — then work FROM the root,
# so relative paths (config/...) resolve exactly as under `python -m src.utils.run_notebooks`.
while not (ROOT / "src" / "visualize").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from src.visualize import exp1_plots as e1, figures, pool_plots as pp, mechanism_plots, prior_plots, style, summaries, literature

style.apply()   # ONE shared style: identical colours in every notebook
pd.set_option("display.width", 200, "display.max_columns", 40)

TASK   = "pd"
CONFIG = "config/Exp1_PD.yaml"   # Exp1 is the prior sweep, so it is the one to visualise
N      = 500     # datasets to draw per prior - enough to show the structure
FOCUS  = None    # variant for the detail plots; None = the first non-original one

# Clears THIS notebook's figure folder - and no other - BEFORE anything is drawn, then saves
# every figure as a PDF sized for A4. Identical in Jupyter and under the batch runner.
FIGS = figures.FigureSaver("0.2_prior_visualisation_pd")

## The colour key

One vocabulary for every figure in every notebook. Blue is our credit prior, grey TabICL's unmodified
prior, orange real data, magenta a real-data marker drawn over coloured points, teal a value from
`tfm-library` and amber one from outside it; the two shades of blue are the mild and aggressive
settings of our prior.

In [ ]:
FIGS.save(
    style.show_palette(),
    "palette",
    caption=(
"Colour key. Grey denotes the unmodified TabICL prior, blue the credit-targeted "
"prior, orange values measured from the real datasets, and red out-of-range or "
"flagged values."
    ),
);


## Loading the priors and the real data

Pre-generated pools if any are on disk, otherwise both arms generated live — the printed `source`
says which. The real PD datasets are loaded alongside: without them every figure would describe the
prior in isolation.

In [ ]:
variants = pp.discover_pools(TASK)
print("pools found:", ", ".join(variants) if variants else "none - will generate live")
loaded, SOURCE = pp.load_variants_or_generate(TASK, n=N, seed=0, config=CONFIG)
print("source =", SOURCE)
print({k: len(v) for k, v in loaded.items()})

FOCUS = FOCUS or next((k for k in loaded if not k.startswith("original")), list(loaded)[0])
print("FOCUS (detail plots) =", FOCUS)

# The real datasets, for every comparison below. Without them the figures describe the prior in
# isolation, which is the thing that made the old notebook uninformative.
REAL = summaries.load_real_datasets(TASK)
print(f"real {TASK.upper()} datasets loaded: {len(REAL)}")
REFERENCE = pp.real_reference(TASK)


## A · Does our prior look like real credit data?

The two priors side by side, against the real books — from the simplest property (the base rate) to the whole target distribution, then the one that defines credit (defaults arriving together).

### A1 · The two priors in numbers

Table shape, target statistics and how each prior sits against the real datasets, before any figure.
A difference in table shape between the priors would be a confound rather than a finding (C4 checks it).

In [ ]:
pp.variant_summary(loaded, TASK)


### A2 · Base rate

The right summary of a binary target is its *rate*, not a histogram of zeros and ones. The real datasets
run from 6.7 % to 40 %; the original prior centres on balance, because nothing in `graph_scm` aims
elsewhere. The teal line is where the literature measured a default-threshold classifier collapsing to
the majority class on real credit data (~10 %, `papers/2026/05_Tanna_DataPresentation` §5.1) — the
regime our prior deliberately reaches into.

**How to read it.** The dashed line is a 50/50 split. The magenta ticks along the bottom are the real datasets' rates; a prior lands well if its histogram overlaps them.

In [ ]:
FIGS.save(
        pp.plot_target_comparison(loaded, TASK, reference=REFERENCE),
        "base_rate_by_variant",
        caption=(
"Distribution of the positive-class rate per synthetic dataset, one step histogram "
"per prior variant, 30 bins. The dashed vertical line marks a 50% rate; dotted lines "
"mark rates measured from the real datasets. Legend gives the mean per variant."
        ),
    );


### A3 · Which prior looks most like real credit data?

For each prior, how far its target distribution sits from each real PD dataset's, as a total-variation
distance — lower is better. The spread of the dots matters as much as the mean: a prior that matches one
dataset and misses the rest is not a good prior. One caution the literature supplies: O'Prior's probe
of Credit-g is a *null* result (0.60–0.66, non-diagnostic; `papers/2026/05_Bouadi_ShapingThePrior`) —
looking realistic is not the same as helping, which is why Experiment 1 trains rather than stopping here.

**How to read it.** One row per prior, best at the top. Grey dots are the individual real datasets and the blue diamond their mean, so a wide spread means the prior matches some datasets and misses others — worse than a slightly larger mean with dots close together.

In [ ]:
FIGS.save(
    e1.plot_prior_realism_ranking(loaded, REAL, task=TASK),
    "prior_realism_ranking",
    caption=(
"Distance between each prior variant's pooled target distribution and each real PD "
"dataset's, as total variation over 40 fixed bins. Diamonds give the mean across "
"datasets, dots one per dataset. Variants ordered by mean distance."
    ),
);


### A4 · Do defaults cluster, as real ones do?

**The PD figure that matters.** Real credit's defining feature is that defaults arrive in *waves*: a
downturn lifts everyone's risk at once, so the default rate varies between cohorts far more than
independent coin flips would. An i.i.d. prior cannot produce that; our Vasicek one-factor mechanism
exists precisely to (B2 shows it at work). The dashed line is the spread chance alone would give, so
*above 1* means genuine clustering rather than noise.

**How to read it.** Left: how much the default rate varies between cohorts, divided by what pure chance would give. Above 1 means defaults genuinely arrive in waves. Right: one dataset per prior — a flat line is independent rows, a jagged one is shared shocks.

In [ ]:
FIGS.save(
        e1.plot_default_clustering(loaded, REAL),
        "default_clustering",
        caption=(
"Left: distribution across synthetic datasets of the between-cohort standard "
"deviation of the default rate, divided by the binomial standard error expected under "
"independence; one violin per prior variant, stars for the real datasets, dashed line "
"at the independence reference. Right: default rate per cohort for one dataset per "
"variant and two real datasets. Cohorts are twelve contiguous blocks of rows."
        ),
    );


## B · Where the difference comes from

Our prior taken apart: each credit mechanism switched on alone, generated live from the Experiment 1 config, and drawn against the original. Six mechanisms, from the one every task gets (a controlled base rate) to the one that decides which tasks are kept at all (the filter).

### B1 · Controlled imbalance

Our prior draws each task's base rate from credit's measured range instead of leaving it to the softmax
bias. The histogram is the per-task default rate under each prior; the orange band is the 7–22 % range
the controlled imbalance targets (the real books themselves run from 6.7 % to 40 %, A2), and the teal
line the ~10 % below which accuracy collapses (`papers/2026/05_Tanna_DataPresentation` §5.1). The
original prior spreads its tasks across the whole [0, 1]; ours concentrates them in credit's range —
including below the collapse line, where a model most needs to have seen tasks.

In [ ]:
FIGS.save(
    mechanism_plots.imbalance_control(CONFIG, n=100),
    'adj_imbalance_control',
    caption=(
        "Per-task positive (default) rate over ~100 tasks from the original prior (grey) and our prior (blue). The shaded band marks the 6.7-22.1% range measured in the real PD datasets."
    ),
);

### B2 · Correlated defaults — the Vasicek factor

With the target base rate fixed at 15 %, a shared systematic factor makes the *realised* rate vary from
task to task: a bad year moves the whole book. Left, the realised rate under mild (ρ 0.03–0.12, retail)
and aggressive (ρ 0.12–0.30, corporate) asset correlation; right, the spread of the realised rate
growing with ρ. The amber line is the Basel IRB corporate cap of 0.24 — **external** domain knowledge,
not a `tfm-library` result — which the aggressive arm reaches past (0.30 = 0.24 × 1.25, the Basel III
large-financial multiplier in `config/Exp1_PD.yaml`).

In [ ]:
FIGS.save(
    mechanism_plots.correlated_defaults(CONFIG, n=80),
    'adj_correlated_defaults',
    caption=(
        "Left: realised default rate over ~80 tasks with the target rate fixed at 15%, under mild versus aggressive asset correlation (SD in the legend). Right: standard deviation of the realised rate against the asset correlation rho, three values."
    ),
);

### B3 · Reject inference

A lender only ever observes the loans it approved. Here the **context** is the approved, lower-risk book
and the **query** reaches into the applicants the screen turned away — so the query is systematically
the riskier book, and every point sits above the diagonal. The mechanism comes from Klein & Hoffart
2026, who argue a statistical prior can only blur a hard underwriting rule (a learned ≈ $4,800 against a
real $5,000 cut-off, `papers/2026/01_Klein_Hoffart`). That is a **position paper with no experiments** —
the most speculative mechanism here, and labelled so.

In [ ]:
FIGS.save(
    mechanism_plots.reject_inference(CONFIG, n=80),
    'adj_reject_inference',
    caption=(
        "Left: query (through-the-door) versus context (approved-book) default rate, one point per task with the selection shift, dashed line y = x. Right: distribution of the query-minus-context rate, dashed line at the mean."
    ),
);

### B4 · Distribution shift — context versus query

Real deployment is never i.i.d.: the loans scored today come from a later cohort, a different mix or a
different screen than the ones the model conditions on. Each panel switches on one kind of shift —
cohort (the whole book drifts), covariate (a feature's range moves), prior probability (the default rate
moves), selection (the reject-inference screen of B3) — and compares context with query. Purucker 2026
shows TFMs lose ground off the i.i.d. regime (`papers/2026/06_Purucker_BeyondIID`), which is why the
prior teaches it.

In [ ]:
FIGS.save(
    mechanism_plots.shift_kinds(CONFIG, n=40),
    'adj_shift_kinds',
    caption=(
        "PD default rate in the context versus query rows, one panel per distribution-shift kind (~40 tasks each). Covariate shift moves the shown feature rather than the rate; selection is the reject-inference shift."
    ),
);

### B5 · Informative missingness (MNAR)

In credit a missing value is itself a signal: a thin file is a risk. Under the MNAR coupling the missing
rate rises with the outcome; under MCAR (coupling 0) it is flat. The unmodified prior injects no
missingness at all and TabICLv2 mean-imputes it away at inference (`repositories/TabICL.txt`
`TransformToNumerical`); O'Prior models MCAR, MAR and MNAR explicitly in its realism engine
(`papers/2026/05_Bouadi_ShapingThePrior` §2.2) — the mechanism this figure reproduces.

In [ ]:
FIGS.save(
    mechanism_plots.informative_missingness(TASK),
    'adj_informative_missingness',
    caption=(
        "Missing rate for non-defaulters versus defaulters, under missing-completely-at-random (grey, coupling 0) and target-coupled missingness (blue, coupling 2)."
    ),
);

### B6 · The predictability filter

Which generated tasks are kept at all. A shallow ExtraTrees (25 trees) is fitted to each candidate task,
and the task is rejected unless it beats the mean at a bootstrap p < 0.05 (`repositories/TabICL.txt`
`should_filter`, `NanoTabICL.txt` `rand_dataset_filtered`). TabICLv2 rejects ~35 % of classification
tasks in stage 1 this way and reports that filtering *improves* convergence
(`papers/2026/02_Qu_TabICLv2` §Data filtering, Fig. 10). But it is a **significance test, not an R²
floor** (§E.14): a weak-but-real signal usually passes. So `banded` targets credit's low-signal band
directly — the shaded region — instead of relying on the filter, a removal that goes against the
published result and is Experiment 1's sharpest test.

In [ ]:
FIGS.save(
    mechanism_plots.filter_modes(CONFIG, n=60),
    'adj_filter_modes',
    caption=(
        "Distribution of ExtraTrees pseudo-R^2 over ~60 generated PD tasks, with the 'banded' keep-region shaded."
    ),
);

## C · Is it a learnable task, and are the tables sane?

The checks that decide whether the prior can teach anything: difficulty, what the model literally sees, the feature structure, and the table shapes.

### C1 · Is the synthetic task the right difficulty?

Invisible in every other figure, and it decides whether the prior teaches anything. A prior whose tasks
are trivially easy teaches the model that features determine the target exactly; one whose tasks are
noise teaches it to predict the mean. Real credit data is neither — **low signal but not zero** — and the
shaded band is that target, measured with the same small-ExtraTrees family the filter (B6) uses.

**How to read it.** The shaded band is where real credit data sits. A prior far above it is too easy and teaches the model that features determine the target almost exactly; far below is noise.

In [ ]:
REAL_SCORES = summaries.real_difficulty(TASK, REAL)
print("real-data difficulty:", {k: round(v, 3) for k, v in REAL_SCORES.items()})

FIGS.save(
    e1.plot_difficulty_calibration(loaded, REAL_SCORES, task=TASK),
    "difficulty_calibration",
    caption=(
"Predictability of each synthetic dataset under a small ExtraTrees on a 70/30 split, "
"one point per dataset and one column per prior variant, with the median marked. The "
"shaded band spans the same measurement on the real credit datasets."
    ),
);


### C2 · What does the model actually see?

Every figure above is a summary statistic. This is the thing itself: one synthetic table and one real
table, same layout, a few rows each. If they obviously differ, no distance rescues the prior; if they do
not, a reader believes the rest more readily.

**How to read it.** Shade is the value's rank within its own column, so compare *texture* — how much variation, how many repeats — not individual cells.

In [ ]:
_real_one = next(iter(REAL.values())) if REAL else None
FIGS.save(
    e1.plot_side_by_side_tables(loaded[FOCUS][0], _real_one, task=TASK),
    "side_by_side_tables",
    caption=(
"Eight rows of one synthetic dataset and one real credit dataset, shown as heatmaps "
"with the target as the final column separated by a vertical rule. Each feature is "
"rank-normalised within its own column, so shade encodes relative value rather than "
"units."
    ),
);


### C3 · Feature dependence structure

O'Prior's central measurement: the eigenvalue spectrum of the feature correlation matrix. Two priors
whose spectra coincide teach a similar dependence structure however different their targets look — the
check that our changes are not *only* about the target.

**How to read it.** Two priors whose curves coincide teach a similar feature-dependence structure, however different their targets look.

In [ ]:
FIGS.save(
    pp.plot_spectrum_by_variant(loaded),
    "spectrum_by_variant",
    caption=(
"Eigenvalue spectra of the feature correlation matrix for up to 40 synthetic datasets "
"per prior variant, normalised by the largest eigenvalue and plotted against "
"normalised eigenvalue rank. Faint lines are individual datasets; bold lines are the "
"per-variant median."
    ),
);


### C4 · Shape sanity check

Rows and features per synthetic dataset, against the real datasets. Cheap, and it catches a
misconfigured prior at once.

**How to read it.** These should MATCH across priors. A difference here is a confound, not a finding.

In [ ]:
FIGS.save(
    pp.plot_shapes_by_variant(loaded),
    "shapes_by_variant",
    caption=(
"Left: distribution of rows per synthetic dataset. Right: distribution of features "
"per synthetic dataset. One step histogram per prior variant, 20 bins."
    ),
);


## Summary

The priors in text: the two priors and their base rates against the real books (A1–A2), then the
realism ranking (A3). Printed last so `output/All_Results.md` carries the numbers, then the
`tfm-library` sources (pin `e5ce016`) and the figure inventory.

In [ ]:
print(summaries.prior_summary(loaded, TASK, source=SOURCE, reference=REFERENCE))
print()
print(summaries.realism_summary(loaded, REAL, TASK))
print()
print(literature.references_md(["imbalance_source", "tanna_paradox", "hc_base_rate", "oprior_headline", "oprior_creditg", "klein_rule", "tabicl_impute", "filter_rate_clf", "filter_pval", "filter_extratrees_n", "merton_vasicek", "basel_corp"]))
print()
print(FIGS.summary())